<a href="https://colab.research.google.com/github/0301sdh/faceguard/blob/main/notebooks/simswap_robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SimSwap 보호 효과 견고성 실험 (FaceGuard Week 6)

Week 5의 pilot(`simswap_eval.ipynb`)을 반복 실험하기 쉽게 정리한 노트북.

**처음 한 번만:** Google Drive에 `MyDrive/faceguard/photos/` 폴더를 만들고 `person_a_1.jpg` ~ `person_a_4.jpg`를 올려 둔다.

**실행 순서:** 런타임을 T4 GPU로 설정 → `준비 1` 실행 (끝나면 런타임이 자동으로 다시 시작됨) → `준비 2` → `도구 함수` → 실험 셀

**결과 저장:** `MyDrive/faceguard/week6/<tag>/`에 바로 저장되므로 세션이 끊겨도 남는다.

출처: [SimSwap](https://github.com/neuralchen/SimSwap) (CC BY-NC 4.0, 학술·비상업 용도). 결과와 해석: `docs/weekly/week6.md`

## 준비

In [ ]:
# [준비 1] SimSwap 받기 + 패키지 설치. 끝나면 런타임이 자동으로 다시 시작된다.
!git clone -q https://github.com/neuralchen/SimSwap /content/SimSwap
# 한 줄로 합치면 pip가 moviepy와 imageio 2.4.1의 버전을 맞추려다 옛 패키지 빌드에 실패함 → Week 5처럼 나눠서 설치
!pip install -q insightface==0.2.1 onnxruntime moviepy
!pip install -q imageio==2.4.1

# 새로 설치한 패키지를 쓰려면 런타임을 다시 시작해야 한다.
# 스스로 종료하면 Colab이 자동으로 다시 시작하므로, 메뉴에서 직접 누를 필요가 없다.
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moviepy 1.0.3 requires imageio<3.0,>=2.5; python_version >= "3.4", but you have imageio 2.4.1 which is incompatible.
scikit-image 0.25.2 requires imageio!=2.35.0,>=2.33, but you have imageio 2.4.1 which is incompatible.


In [ ]:
# [준비 2] 다시 시작된 뒤 실행. 가중치 받기 + 옛 코드 수정 + 모델 불러오기 + Drive 연결
import os, io, contextlib
os.chdir('/content/SimSwap')

# 1) 가중치 (이미 있으면 건너뜀)
if not os.path.exists('arcface_model/arcface_checkpoint.tar'):
    !wget -q -P ./arcface_model https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar
    !wget -q https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
    !unzip -q ./checkpoints.zip -d ./checkpoints
    !wget -q -P ./parsing_model/checkpoint https://github.com/neuralchen/SimSwap/releases/download/1.0/79999_iter.pth

# 2) 얼굴 검출 모델: 노트북의 OneDrive 링크가 만료되어 insightface 공식 릴리스에서 받음
if not os.path.exists('insightface_func/models/antelope/scrfd_10g_bnkps.onnx'):
    !wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip -O /content/antelopev2.zip
    !unzip -o -q /content/antelopev2.zip -d /content/antelopev2_tmp
    !mkdir -p insightface_func/models/antelope
    !cp $(find /content/antelopev2_tmp -name scrfd_10g_bnkps.onnx) insightface_func/models/antelope/

# 3) 옛 코드를 최신 환경에 맞게 수정 (Week 5 트러블슈팅)
#    - ArcFace 파일은 모델 객체 전체를 저장한 것 → PyTorch 2.6+에서 weights_only=False 필요
#    - numpy에서 없어진 np.float, np.int를 다시 연결
!sed -i 's/map_location=torch.device("cpu"))/map_location=torch.device("cpu"), weights_only=False)/' models/fs_model.py
import numpy as np
np.float = float
np.int = int

import cv2
import torch
import torch.nn.functional as F
from models.models import create_model
from options.test_options import TestOptions
from insightface_func.face_detect_crop_multi import Face_detect_crop
from util.reverse2original import reverse2wholeimage
from util.norm import SpecificNorm
from parsing_model.model import BiSeNet

# 4) 모델 불러오기
opt = TestOptions()
opt.initialize()
opt.parser.add_argument('-f')                    # Colab에서 생기는 인자 오류 방지
with contextlib.redirect_stdout(io.StringIO()):  # 긴 옵션 목록 출력 숨김
    opt = opt.parse()
opt.Arc_path = './arcface_model/arcface_checkpoint.tar'
opt.isTrain = False
opt.crop_size = 224                              # 받은 가중치(people)가 224용
crop_size = opt.crop_size

torch.nn.Module.dump_patches = True
model = create_model(opt)                        # SimSwap 본체 + ArcFace(model.netArc)
model.eval()

app = Face_detect_crop(name='antelope', root='./insightface_func/models')   # 얼굴 찾기·정렬
app.prepare(ctx_id=0, det_thresh=0.6, det_size=(640, 640))

net = BiSeNet(n_classes=19).cuda()               # 합성한 얼굴을 자연스럽게 붙이는 parsing 모델
net.load_state_dict(torch.load('./parsing_model/checkpoint/79999_iter.pth'))
net.eval()
spNorm = SpecificNorm()

# 5) Google Drive: 내 사진을 읽고, 결과를 저장 (세션이 끊겨도 남음)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/faceguard'
PHOTOS = f'{DRIVE}/photos'
RESULTS = f'{DRIVE}/week6'
os.makedirs(RESULTS, exist_ok=True)

names = ['person_a_1', 'person_a_2', 'person_a_3', 'person_a_4']
target_path = './demo_file/specific1.png'
missing = [n for n in names if not os.path.exists(f'{PHOTOS}/{n}.jpg')]
print('준비 완료' if not missing else f'Drive에 사진이 없음: {missing}')

find model: ./insightface_func/models/antelope/scrfd_10g_bnkps.onnx detection
set det-size: (640, 640)
Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to /root/.cache/torch/hub/checkpoints/resnet18-5c106cde.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 277MB/s]


Mounted at /content/drive
준비 완료


## 도구 함수
세션을 새로 열 때마다 이 구역의 셀 3개를 모두 실행한다. 함수를 정의만 하므로 바로 끝난다.

In [ ]:
# [도구 함수] Week 5에서 쓴 함수들을 한곳에 모음
import random
mean = torch.tensor([0.485, 0.456, 0.406], device='cuda').view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device='cuda').view(1, 3, 1, 1)

# ---------- 사진 ↔ 텐서 ----------
def to_tensor(img_bgr):
    # OpenCV 사진(BGR, 0~255) → (1, 3, H, W) RGB 텐서, 0~1
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float().div(255)[None].cuda()

def save_png(x, path):
    img = (x[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
    cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

def add_margin(img_bgr, pad=112):
    # 얼굴만 꽉 찬 224×224 사진 둘레에 검은 여백 → 검출 모델이 얼굴을 찾기 쉽게
    return cv2.copyMakeBorder(img_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0, 0, 0))

# ---------- ArcFace 명세서 ----------
def embed(x):
    # x: (1, 3, 224, 224), RGB, 0~1. SimSwap과 같은 순서: 정규화 → 112×112 → ArcFace → 길이 1
    x = x.contiguous()   # permute로 만든 텐서는 메모리가 순서대로 놓여 있지 않아 ArcFace의 view()가 실패함
    x = (x - mean) / std
    x = F.interpolate(x, size=(112, 112))
    return F.normalize(model.netArc(x), p=2, dim=1)

def load_crop(img_bgr):
    # SimSwap이 정렬해서 잘라낸 224×224 얼굴. 얼굴을 못 찾으면 None
    found = app.get(img_bgr, crop_size)
    return None if found is None else to_tensor(found[0][0])

def arc_embed_img(img_bgr):
    # 사진 → (얼굴 찾기·정렬) → ArcFace 명세서. 얼굴을 못 찾으면 None
    face = load_crop(img_bgr)
    if face is None:
        return None
    with torch.no_grad():
        return embed(face)

# ---------- PGD (랜덤 출발 + 4가지 이동 평균) ----------
SHIFTS = [(0, 0), (0, 1), (1, 0), (1, 1)]   # 2×2 중 어느 칸이 뽑혀도 통하도록

def shift(x, dy, dx):
    return torch.roll(x, shifts=(dy, dx), dims=(2, 3))

def pgd_identity(face, epsilon, steps=30):
    alpha = 2.5 * epsilon / steps
    orig_embs = [embed(shift(face, dy, dx)).detach() for dy, dx in SHIFTS]
    # 원본(유사도 1.0, 기울기 0)에서 출발하면 움직이지 못하므로 ±epsilon 안의 임의 지점에서 출발
    adv = (face + torch.empty_like(face).uniform_(-epsilon, epsilon)).clamp(0, 1).detach()
    for _ in range(steps):
        adv = adv.detach().requires_grad_(True)
        sims = [F.cosine_similarity(embed(shift(adv, dy, dx)), orig).mean()
                for (dy, dx), orig in zip(SHIFTS, orig_embs)]
        loss = sum(sims) / len(sims)
        model.netArc.zero_grad()
        loss.backward()
        adv = adv - alpha * adv.grad.sign()          # 유사도를 낮추는 방향으로 이동
        perturbation = torch.clamp(adv - face, -epsilon, epsilon)
        adv = torch.clamp(face + perturbation, 0, 1)
    return adv.detach()

# ---------- SimSwap 합성 ----------
def swap_with_latent(latend_id, output_path):
    # 명세서를 받아 target 사진의 얼굴을 다시 그리고 저장
    img_b_whole = cv2.imread(target_path)
    crops_b, mats_b = app.get(img_b_whole, crop_size)
    tensors_b, results = [], []
    with torch.no_grad():
        for b_crop in crops_b:
            b = to_tensor(b_crop)
            results.append(model(None, b, latend_id, None, True)[0])
            tensors_b.append(b)
        reverse2wholeimage(tensors_b, results, mats_b, crop_size, img_b_whole, None,
                           output_path, no_simswaplogo=True, pasring_model=net, use_mask=True, norm=spNorm)

# ---------- 판정 (Week 5에서 정한 ArcFace 기준) ----------
ARC_FAIL = 0.406      # 진짜 내 사진끼리의 최솟값. 이 이상이면 나로 볼 수 있음
ARC_SUCCESS = 0.2     # 본인 최소와 타인(약 0)의 중간. 이 아래면 보호 성공

def judge(score):
    if score is None:
        return '검출 실패'
    if score >= ARC_FAIL:
        return '실패'
    return '부분' if score >= ARC_SUCCESS else '성공'

# ---------- 실험 한 번 ----------
def run(tag, eps, steps=30, attack=pgd_identity, seed=0):          # ← attack 인자 추가
    # 보호 사진 만들기 → (B) 재정렬로 SimSwap 합성 → 결과 vs 내 사진 (ArcFace) 채점
    # 결과는 Drive의 week6/<tag>/ 에 저장. eps=0이면 노이즈 없이 baseline
    random.seed(seed)          # 무작위 변형(회전·크기·이동)을 재현 가능하게
    torch.manual_seed(seed)    # 랜덤 출발을 재현 가능하게
    out_dir = f'{RESULTS}/{tag}'
    os.makedirs(out_dir, exist_ok=True)
    scores = {}
    for name in names:
        photo = cv2.imread(f'{PHOTOS}/{name}.jpg')
        me = arc_embed_img(photo)                          # 채점 기준: 원본 내 사진
        face = load_crop(photo)
        adv = face if eps == 0 else attack(face, eps / 255, steps)   # ← pgd_identity 대신 attack
        png = f'{out_dir}/{name}_eps{eps}.png'
        save_png(adv, png)                                 # 보호 얼굴 (PNG, 무손실)

        lat = arc_embed_img(add_margin(cv2.imread(png)))   # SimSwap이 다시 찾고 정렬한 명세서
        score = None
        if lat is not None:
            result_path = f'{out_dir}/{name}_eps{eps}_realign.jpg'
            swap_with_latent(lat, result_path)
            score = (arc_embed_img(cv2.imread(result_path)) * me).sum().item()
        scores[name] = score
        shown = '-' if score is None else f'{score:.3f}'
        print(f'[{tag}] {name} eps={eps:>2}/255 steps={steps} | (B) 재정렬 결과 vs 내 사진 (ArcFace) {shown} ({judge(score)})')
    return scores

print('도구 함수 준비 완료')

도구 함수 준비 완료


In [ ]:
# [2단계] 회전·크기·이동 EOT
import math, random

def random_theta(max_deg=3, max_scale=0.05, max_shift=2, size=224):
    # 재정렬이 만드는 작은 변형을 흉내: 회전 ±3도, 크기 ±5%, 이동 ±2픽셀 (무작위)
    angle = math.radians(random.uniform(-max_deg, max_deg))
    scale = 1 + random.uniform(-max_scale, max_scale)
    tx = random.uniform(-max_shift, max_shift) * 2 / size   # affine_grid는 좌표를 -1~1로 쓰므로 픽셀을 비율로 바꿈
    ty = random.uniform(-max_shift, max_shift) * 2 / size
    c, s = math.cos(angle) / scale, math.sin(angle) / scale
    return torch.tensor([[c, -s, tx], [s, c, ty]], device='cuda').unsqueeze(0)

def warp(x, theta):
    # theta대로 사진을 회전·확대·이동. 미분 가능한 연산이라 기울기가 원래 픽셀까지 전달됨
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)

def pgd_identity_eot(face, epsilon, steps=30, n_samples=4):
    # 랜덤 출발 PGD + 반복마다 무작위 변형 n_samples개의 평균 유사도를 낮춤
    alpha = 2.5 * epsilon / steps
    adv = (face + torch.empty_like(face).uniform_(-epsilon, epsilon)).clamp(0, 1).detach()
    for _ in range(steps):
        adv = adv.detach().requires_grad_(True)
        sims = []
        for k in range(n_samples):
            if k == 0:   # 하나는 변형 없이 (재정렬이 거의 없는 경우도 지키기 위해)
                x_adv, x_face = adv, face
            else:        # 보호 얼굴과 원본 얼굴에 같은 변형을 적용 → 변형 자체의 영향은 빼고 노이즈 효과만 봄
                theta = random_theta()
                x_adv, x_face = warp(adv, theta), warp(face, theta)
            with torch.no_grad():
                orig = embed(x_face)
            sims.append(F.cosine_similarity(embed(x_adv), orig).mean())
        loss = sum(sims) / len(sims)
        model.netArc.zero_grad()
        loss.backward()
        adv = adv - alpha * adv.grad.sign()
        perturbation = torch.clamp(adv - face, -epsilon, epsilon)
        adv = torch.clamp(face + perturbation, 0, 1)
    return adv.detach()

print('EOT 함수 준비 완료')

EOT 함수 준비 완료


In [ ]:
# [5단계] 변형 견고성: 보호 사진에 현실의 변형을 적용한 뒤 SimSwap
import csv
from collections import defaultdict

# ---------- 변형 (입력·출력 모두 OpenCV 사진, BGR, 0~255) ----------
def jpeg(img, quality):
    # SNS 업로드처럼 JPEG로 압축했다가 다시 읽음
    ok, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)

def resize(img, scale, interp):
    # 작게 줄였다가 원래 크기로 되돌림 (썸네일, 크기 조절)
    h, w = img.shape[:2]
    small = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=interp)
    return cv2.resize(small, (w, h), interpolation=interp)

def crop(img, frac):
    # 가장자리를 잘라냄 (위아래 좌우 각각 frac/2씩)
    h, w = img.shape[:2]
    dy, dx = int(h * frac / 2), int(w * frac / 2)
    return img[dy:h - dy, dx:w - dx]

def blur(img, sigma):
    return cv2.GaussianBlur(img, (0, 0), sigma)

TRANSFORMS = {
    'none':          lambda im: im,
    'jpeg90':        lambda im: jpeg(im, 90),
    'jpeg75':        lambda im: jpeg(im, 75),
    'jpeg50':        lambda im: jpeg(im, 50),
    'resize75_bili': lambda im: resize(im, 0.75, cv2.INTER_LINEAR),
    'resize50_bili': lambda im: resize(im, 0.50, cv2.INTER_LINEAR),
    'resize75_area': lambda im: resize(im, 0.75, cv2.INTER_AREA),
    'resize50_area': lambda im: resize(im, 0.50, cv2.INTER_AREA),
    'crop10':        lambda im: crop(im, 0.10),
    'blur0.5':       lambda im: blur(im, 0.5),
    'blur1.0':       lambda im: blur(im, 1.0),
}

# ---------- 실험: 보호 사진은 seed마다 한 번만 만들고, 변형을 각각 적용 ----------
def run_transforms(eps, seeds=(0, 1, 2), attack=pgd_identity_eot, steps=30):
    rows = []
    settings = [(0, None)] + [(eps, s) for s in seeds]     # 보호 안 함(무작위 없음, 한 번) + 보호(seed마다)
    for e, seed in settings:
        tag = 'robust_eps0' if e == 0 else f'robust_eps{e}_seed{seed}'
        out_dir = f'{RESULTS}/{tag}'
        os.makedirs(out_dir, exist_ok=True)
        if seed is not None:
            random.seed(seed)
            torch.manual_seed(seed)
        print(f'[{tag}] 시작')
        for name in names:
            photo = cv2.imread(f'{PHOTOS}/{name}.jpg')
            me = arc_embed_img(photo)                                   # 채점 기준: 원본 내 사진
            face = load_crop(photo)
            adv = face if e == 0 else attack(face, e / 255, steps)
            png = f'{out_dir}/{name}_eps{e}.png'
            save_png(adv, png)
            protected = cv2.imread(png)

            for tname, transform in TRANSFORMS.items():
                img = transform(protected)                              # 변형 적용
                lat = arc_embed_img(add_margin(img))                    # SimSwap이 다시 찾고 정렬한 명세서
                score = None
                if lat is not None:
                    result_path = f'{out_dir}/{name}_eps{e}_{tname}_realign.jpg'
                    swap_with_latent(lat, result_path)
                    res = arc_embed_img(cv2.imread(result_path))
                    if res is not None:
                        score = (res * me).sum().item()
                rows.append({'tag': tag, 'eps': e, 'seed': seed, 'name': name, 'transform': tname, 'arcface': score})
    return rows

# ---------- 요약: 변형별로 보호 안 함 vs 보호, 그리고 효과가 얼마나 남았나 ----------
def summarize(rows):
    scores = defaultdict(list)
    for r in rows:
        if r['arcface'] is not None:
            scores[(r['eps'], r['transform'])].append(r['arcface'])
    eps = max(r['eps'] for r in rows)
    gap_none = np.mean(scores[(0, 'none')]) - np.mean(scores[(eps, 'none')])   # 변형 없을 때의 보호 효과
    print(f'\n=== ArcFace (B) 재정렬, 결과 vs 내 사진 (사진 4장, 보호는 seed 평균) ===')
    print(f'{"변형":<15}{"보호 안 함":>10}{"보호":>9}{"차이":>8}{"유지율":>8}   판정(보호)')
    for tname in TRANSFORMS:
        a, b = np.mean(scores[(0, tname)]), np.mean(scores[(eps, tname)])
        missing = sum(1 for r in rows if r['transform'] == tname and r['arcface'] is None)
        note = f'  (얼굴 검출 실패 {missing}개)' if missing else ''
        print(f'{tname:<15}{a:>10.3f}{b:>9.3f}{a - b:>8.3f}{(a - b) / gap_none:>8.0%}   {judge(b)}{note}')

def save_csv(rows, path):
    with open(path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

print('5단계 함수 준비 완료')

5단계 함수 준비 완료


## 실험 기록
각 실험은 독립적으로 실행할 수 있다. 결과와 해석은 `docs/weekly/week6.md` 참고.

### 확인: Week 5 결과가 다시 나오는가

- `eps=0`: 보호 안 한 baseline → 0.68 ~ 0.77 근처 (Week 5)
- `eps=8`: 랜덤 출발 + 이동 평균 → -0.18 ~ 0.09 근처 (Week 5). 랜덤 출발 때문에 정확히 같지는 않음

In [ ]:
baseline = run('check', 0)
protected8 = run('check', 8)

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[check] person_a_1 eps= 0/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.774 (실패)
[check] person_a_2 eps= 0/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.684 (실패)
[check] person_a_3 eps= 0/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.699 (실패)
[check] person_a_4 eps= 0/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.733 (실패)
[check] person_a_1 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.138 (성공)
[check] person_a_2 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.025 (성공)
[check] person_a_3 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.062 (성공)
[check] person_a_4 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.115 (성공)


### 이동만 vs 회전·크기 EOT

In [ ]:
shift8   = run('shift_eps8', 8)                                    # Week 5 방식 (이동만)
affine8  = run('affine_eps8', 8, attack=pgd_identity_eot)          # 회전·크기·이동
shift12  = run('shift_eps12', 12)
affine12 = run('affine_eps12', 12, attack=pgd_identity_eot)

[shift_eps8] person_a_1 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.037 (성공)
[shift_eps8] person_a_2 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.086 (성공)
[shift_eps8] person_a_3 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.032 (성공)
[shift_eps8] person_a_4 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.079 (성공)
[affine_eps8] person_a_1 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.369 (성공)
[affine_eps8] person_a_2 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.425 (성공)
[affine_eps8] person_a_3 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.290 (성공)
[affine_eps8] person_a_4 eps= 8/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.343 (성공)
[shift_eps12] person_a_1 eps=12/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.159 (성공)
[shift_eps12] person_a_2 eps=12/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.291 (성공)
[shift_eps12] person_a_3 eps=12/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.188 (성공)
[shift_eps12] person_a_4 eps=12/255 s

###  eps, steps 비교

In [ ]:
run('affine_eps4',          4,           attack=pgd_identity_eot)
run('affine_eps4_steps50',  4, steps=50, attack=pgd_identity_eot)
run('affine_eps8_steps50',  8, steps=50, attack=pgd_identity_eot)
run('affine_eps12_steps50', 12, steps=50, attack=pgd_identity_eot)

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[affine_eps4] person_a_1 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.192 (성공)
[affine_eps4] person_a_2 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.149 (성공)
[affine_eps4] person_a_3 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.149 (성공)
[affine_eps4] person_a_4 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.131 (성공)
[affine_eps4_steps50] person_a_1 eps= 4/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.161 (성공)
[affine_eps4_steps50] person_a_2 eps= 4/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.216 (성공)
[affine_eps4_steps50] person_a_3 eps= 4/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.116 (성공)
[affine_eps4_steps50] person_a_4 eps= 4/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.125 (성공)
[affine_eps8_steps50] person_a_1 eps= 8/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.414 (성공)
[affine_eps8_steps50] person_a_2 eps= 8/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.376 (성공)
[affine_eps8_steps50] person_a_3 eps= 8/255 steps=50 | (B) 재정렬 결과 vs 내 사진 (A

{'person_a_1': -0.41621488332748413,
 'person_a_2': -0.5233050584793091,
 'person_a_3': -0.4185536503791809,
 'person_a_4': -0.4485543668270111}

### eps 5·6, EOT 변형 8개

In [ ]:
# eps 크기 조정: 4와 8 사이
eps5 = run('affine_eps5', 5, attack=pgd_identity_eot)
eps6 = run('affine_eps6', 6, attack=pgd_identity_eot)

# ③-2 A: eps 4에서 EOT 변형 4개 vs 8개
eot4 = run('affine_eps4_n4', 4, attack=pgd_identity_eot)
eot8 = run('affine_eps4_n8', 4, attack=lambda f, e, s: pgd_identity_eot(f, e, s, n_samples=8))

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[affine_eps5] person_a_1 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.179 (성공)
[affine_eps5] person_a_2 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.167 (성공)
[affine_eps5] person_a_3 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.008 (성공)
[affine_eps5] person_a_4 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.080 (성공)
[affine_eps6] person_a_1 eps= 6/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.243 (성공)
[affine_eps6] person_a_2 eps= 6/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.387 (성공)
[affine_eps6] person_a_3 eps= 6/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.065 (성공)
[affine_eps6] person_a_4 eps= 6/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.180 (성공)
[affine_eps4_n4] person_a_1 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.137 (성공)
[affine_eps4_n4] person_a_2 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.082 (성공)
[affine_eps4_n4] person_a_3 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.151 (성공)
[affine_eps4_n4] person_

### 반복 실행 (seed 3개)

In [ ]:
# ④ 반복 실행: 후보 3개 × seed 3개
n8 = lambda f, e, s: pgd_identity_eot(f, e, s, n_samples=8)   # 변형 8개 버전
for seed in [0, 1, 2]:
    run(f'eps4_n4_seed{seed}', 4, attack=pgd_identity_eot, seed=seed)
    run(f'eps4_n8_seed{seed}', 4, attack=n8, seed=seed)
    run(f'eps5_n4_seed{seed}', 5, attack=pgd_identity_eot, seed=seed)

[eps4_n4_seed0] person_a_1 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.128 (성공)
[eps4_n4_seed0] person_a_2 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.064 (성공)
[eps4_n4_seed0] person_a_3 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.116 (성공)
[eps4_n4_seed0] person_a_4 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.060 (성공)
[eps4_n8_seed0] person_a_1 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.098 (성공)
[eps4_n8_seed0] person_a_2 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.079 (성공)
[eps4_n8_seed0] person_a_3 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.054 (성공)
[eps4_n8_seed0] person_a_4 eps= 4/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) 0.053 (성공)
[eps5_n4_seed0] person_a_1 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.260 (성공)
[eps5_n4_seed0] person_a_2 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.158 (성공)
[eps5_n4_seed0] person_a_3 eps= 5/255 steps=30 | (B) 재정렬 결과 vs 내 사진 (ArcFace) -0.026 (성공)
[eps5_n4_seed0

### 변형 견고성  

In [ ]:
# [5단계] eps 5/255, 변형 4개, steps 30 / seed 0, 1, 2
rows = run_transforms(5, seeds=(0, 1, 2))
save_csv(rows, f'{RESULTS}/robust_arcface.csv')   # 그래프용으로 Drive에 저장
summarize(rows)

[robust_eps0] 시작
[robust_eps5_seed0] 시작
[robust_eps5_seed1] 시작
[robust_eps5_seed2] 시작

=== ArcFace (B) 재정렬, 결과 vs 내 사진 (사진 4장, 보호는 seed 평균) ===
변형                 보호 안 함       보호      차이     유지율   판정(보호)
none                0.723   -0.128   0.851    100%   성공
jpeg90              0.723   -0.107   0.830     98%   성공
jpeg75              0.714   -0.095   0.810     95%   성공
jpeg50              0.717   -0.076   0.793     93%   성공
resize75_bili       0.721   -0.108   0.829     97%   성공
resize50_bili       0.714   -0.097   0.811     95%   성공
resize75_area       0.722   -0.111   0.832     98%   성공
resize50_area       0.724   -0.097   0.821     97%   성공
crop10              0.714   -0.096   0.809     95%   성공
blur0.5             0.721   -0.116   0.838     98%   성공
blur1.0             0.718   -0.100   0.818     96%   성공


### Week 7 ensemble 준비: FaceID의 ArcFace를 공격할 수 있게 만들기
- Week 7 pilot에서 이 노트북의 보호 사진은 FaceID(insightface ArcFace)에 **부분적으로만** 통했다 (`docs/weekly/week7.md`)
- 다음 단계는 SimSwap ArcFace와 insightface ArcFace를 **함께** 속이는 ensemble 공격
- insightface ArcFace는 `.onnx`(계산 결과만 주는 완성품)라 gradient가 없음 → PyTorch로 변환
- **확인 1:** 변환한 모델이 원래 onnx와 같은 값을 내는가 (유사도 0.999 이상)
- **확인 2:** 변환한 모델에서 gradient가 흘러나오는가 (0이 아님)
- 실행 순서: `준비 1` → `준비 2` → `도구 함수` → 아래 셀


In [ ]:
# [ensemble 준비 1] FaceID의 ArcFace(insightface, .onnx) → PyTorch 변환 + 검증
# .onnx는 계산 결과만 내주는 완성품이라 gradient(어느 방향으로 픽셀을 바꿔야 하는지)를 못 줌
# → PyTorch 모델로 바꿔야 PGD로 공격할 수 있다
!pip install -q onnx2torch
if not os.path.exists('/content/buffalo_l'):
    !wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip -O /content/buffalo_l.zip
    !unzip -o -q /content/buffalo_l.zip -d /content/buffalo_l

import glob, onnx, onnxruntime as ort
from onnx2torch import convert

FID_ONNX = glob.glob('/content/buffalo_l/**/w600k_r50.onnx', recursive=True)[0]   # FaceID가 쓰는 ArcFace
fid_arc = convert(onnx.load(FID_ONNX)).eval().cuda()   # PyTorch 버전
for p in fid_arc.parameters():
    p.requires_grad_(False)                           # 모델은 고정, 사진만 바꿀 것
sess = ort.InferenceSession(FID_ONNX, providers=['CPUExecutionProvider'])   # 원래 onnx (정답지)
print('입력 모양:', sess.get_inputs()[0].shape)

def fid_input(x):
    # x: (N, 3, H, W) RGB 0~1 → insightface 규칙: 112×112, (픽셀-127.5)/127.5 = -1~1
    x = F.interpolate(x, size=(112, 112), mode='bilinear', align_corners=False)
    return (x - 0.5) / 0.5

# 같은 사진을 두 버전에 넣고 결과가 같은지 비교
for n in names:
    x = fid_input(to_tensor(cv2.imread(f'{RESULTS}/robust_eps0/{n}_eps0.png')))
    out_onnx = torch.from_numpy(sess.run(None, {sess.get_inputs()[0].name: x.cpu().numpy()})[0])
    with torch.no_grad():
        out_torch = fid_arc(x).cpu()
    c = F.cosine_similarity(out_onnx, out_torch).item()
    print(f'{n} | onnx vs PyTorch 유사도 {c:.5f} ({"통과" if c > 0.999 else "확인 필요"})')


In [ ]:
# [ensemble 준비 2] 변환한 모델에서 gradient가 실제로 흘러나오는가
x = to_tensor(cv2.imread(f'{RESULTS}/robust_eps0/person_a_1_eps0.png'))
with torch.no_grad():
    orig = F.normalize(fid_arc(fid_input(x)), dim=1)               # 원본 명세서

# 원본 근처에서 출발 (원본 그대로면 유사도 1.0이라 기울기 0 → Week 5와 같은 이유)
adv = (x + torch.empty_like(x).uniform_(-5/255, 5/255)).clamp(0, 1).requires_grad_(True)
sim = F.cosine_similarity(F.normalize(fid_arc(fid_input(adv)), dim=1), orig).mean()
sim.backward()
g = adv.grad
print(f'유사도 {sim.item():.3f} | gradient 평균 크기 {g.abs().mean().item():.2e} | 0인 비율 {(g == 0).float().mean().item():.1%}')


**확인 1·2 결과:** onnx vs PyTorch 4장 모두 1.00000, gradient 평균 4.46e-05·0인 비율 0.0% → 변환 성공, 공격 가능

#### 확인 3: FaceID가 실제로 보는 112×112 얼굴을 똑같이 만들기
- FaceID는 보호 얼굴(224)을 그냥 줄이지 않는다: 여백 붙이기 → 얼굴 찾기 → 눈·코·입꼬리 5점 → **insightface 기준 위치**로 맞춰 112×112로 자름
- SimSwap의 224 crop은 기준 위치가 조금 다르다 (눈 사이 간격·입 위치가 다름) → 단순히 112로 줄이면 FaceID가 보는 얼굴과 크기·위치가 어긋남
- 그래서 같은 변환(M)을 **미분 가능한 방식**(grid_sample)으로 따라 해서, gradient가 FaceID가 실제로 보는 얼굴에 대해 계산되게 한다
- 통과 기준: 따라 한 버전 vs 실제 FaceID 방식 유사도 0.999 이상. 참고로 단순 축소 버전이 얼마나 어긋나는지도 출력


In [ ]:
# [ensemble 준비 3] FaceID의 정렬(얼굴 찾기 → 5점 → 112 기준 위치)을 미분 가능하게 따라 하기
from skimage import transform as trans

# insightface가 112×112에서 쓰는 기준 위치: 왼눈, 오른눈, 코, 왼입꼬리, 오른입꼬리
ARCFACE_DST = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
                        [41.5493, 92.3655], [70.7299, 92.2041]], dtype=np.float32)

def fid_matrix(crop_bgr, pad=112):
    # 보호 얼굴(224)에 여백 → 얼굴 찾기 → 5점 → 112 기준 위치로 옮기는 변환 M (2×3, 224 crop 좌표 기준)
    bboxes, kpss = app.det_model.detect(add_margin(crop_bgr, pad), threshold=0.6, max_num=0, metric='default')
    if bboxes.shape[0] == 0:
        return None
    i = np.argmax((bboxes[:, 2] - bboxes[:, 0]) * (bboxes[:, 3] - bboxes[:, 1]))   # 가장 큰 얼굴
    t = trans.SimilarityTransform()
    t.estimate(kpss[i] - pad, ARCFACE_DST)   # 여백만큼 빼서 224 crop 좌표로
    return t.params[:2].astype(np.float32)

def fid_align(x, M):
    # x: (N, 3, 224, 224) RGB 0~1 → cv2.warpAffine(x, M, (112, 112))과 같은 결과를 grid_sample로 (gradient가 흐름)
    Minv = torch.from_numpy(cv2.invertAffineTransform(M)).cuda()        # 112의 각 픽셀이 224의 어디서 오는가
    v, u = torch.meshgrid(torch.arange(112., device='cuda'), torch.arange(112., device='cuda'), indexing='ij')
    src = torch.stack([u, v, torch.ones_like(u)], -1) @ Minv.T          # (112, 112, 2) 224 좌표
    H, W = x.shape[2:]
    grid = torch.stack([2 * src[..., 0] / (W - 1) - 1, 2 * src[..., 1] / (H - 1) - 1], -1)   # -1~1로
    return F.grid_sample(x, grid[None].expand(x.shape[0], -1, -1, -1), mode='bilinear',
                         padding_mode='zeros', align_corners=True)     # 밖은 검정 (cv2와 같음)

def fid_embed(x, M):
    # FaceID가 생성 모델에 넘기는 명세서와 같은 값 (길이 1인 512개 숫자)
    return F.normalize(fid_arc((fid_align(x, M) - 0.5) / 0.5), dim=1)

FID_M = {}
for n in names:
    crop = cv2.imread(f'{RESULTS}/robust_eps0/{n}_eps0.png')
    M = fid_matrix(crop)
    if M is None:
        print(f'{n} | 얼굴 못 찾음'); continue
    FID_M[n] = M
    x = to_tensor(crop)
    # 정답: FaceID 방식 그대로 (cv2로 자르기 → onnx)
    real = cv2.cvtColor(cv2.warpAffine(crop, M, (112, 112), borderValue=0.0), cv2.COLOR_BGR2RGB)
    real = ((real.astype(np.float32) / 255 - 0.5) / 0.5).transpose(2, 0, 1)[None]
    e_real = F.normalize(torch.from_numpy(sess.run(None, {sess.get_inputs()[0].name: real})[0]), dim=1)
    with torch.no_grad():
        e_ours = fid_embed(x, M).cpu()
        e_resize = F.normalize(fid_arc(fid_input(x)), dim=1).cpu()
    c1 = F.cosine_similarity(e_real, e_ours).item()
    c2 = F.cosine_similarity(e_real, e_resize).item()
    print(f'{n} | 따라 한 정렬 {c1:.4f} ({"통과" if c1 > 0.999 else "확인 필요"}) | 참고: 단순 축소 {c2:.4f}')


**확인 3 결과:** 따라 한 정렬 0.9998~0.9999 (4장 통과), 단순 축소는 0.936~0.965 → 그냥 줄이면 FaceID가 보는 얼굴과 어긋남. 공격에는 따라 한 정렬(`fid_embed`)을 쓴다

### Week 7 ensemble 공격: SimSwap ArcFace + FaceID ArcFace를 함께 속이기
- Week 6 최종 설정 그대로 (eps 5/255, EOT 변형 4개, steps 30, seed 0). **loss만** 두 모델 유사도의 합으로 바꿈
  - `loss = SimSwap ArcFace 유사도 + FaceID ArcFace 유사도` → 둘 다 낮추는 방향으로 노이즈를 만든다
- 결과: Drive `week6/ens_eps5_n4_seed0/` (보호 얼굴 + SimSwap 합성). 셀이 SimSwap 채점까지 출력
- 다음 셀: 생성 없이 **입력 단계**에서 FaceID ArcFace가 보호 얼굴을 원본과 얼마나 같게 보는지 pilot과 비교 (빠른 사전 확인)
- 그다음 FaceID 노트북에서 생성 + 채점 (`ipadapter_faceid_eval.ipynb`)


In [ ]:
# [ensemble 공격] Week 6 EOT PGD와 같고, loss만 SimSwap ArcFace + FaceID ArcFace
def tensor_to_bgr(x):
    img = (x[0].permute(1, 2, 0).detach().cpu().numpy() * 255).round().astype(np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

def pgd_ensemble_eot(face, epsilon, steps=30, n_samples=4, w_fid=1.0):
    # w_fid: FaceID 쪽 비중 (1.0 = 두 모델을 똑같이)
    M = fid_matrix(tensor_to_bgr(face))   # FaceID가 이 얼굴을 자르는 위치 (원본 얼굴로 한 번만 구함)
    assert M is not None, 'FaceID 방식으로 얼굴을 못 찾음'
    alpha = 2.5 * epsilon / steps
    adv = (face + torch.empty_like(face).uniform_(-epsilon, epsilon)).clamp(0, 1).detach()
    for _ in range(steps):
        adv = adv.detach().requires_grad_(True)
        sims_sw, sims_fid = [], []
        for k in range(n_samples):
            if k == 0:
                x_adv, x_face = adv, face
            else:        # 보호 얼굴과 원본 얼굴에 같은 무작위 변형 (Week 6과 같음)
                theta = random_theta()
                x_adv, x_face = warp(adv, theta), warp(face, theta)
            with torch.no_grad():
                orig_sw, orig_fid = embed(x_face), fid_embed(x_face, M)
            sims_sw.append(F.cosine_similarity(embed(x_adv), orig_sw).mean())            # SimSwap이 보는 신원
            sims_fid.append(F.cosine_similarity(fid_embed(x_adv, M), orig_fid).mean())   # FaceID가 보는 신원
        loss = sum(sims_sw) / n_samples + w_fid * sum(sims_fid) / n_samples
        model.netArc.zero_grad()
        loss.backward()
        adv = adv - alpha * adv.grad.sign()          # 두 유사도의 합을 낮추는 방향
        perturbation = torch.clamp(adv - face, -epsilon, epsilon)
        adv = torch.clamp(face + perturbation, 0, 1)
    return adv.detach()

ens = run('ens_eps5_n4_seed0', 5, attack=pgd_ensemble_eot, seed=0)


In [ ]:
# [사전 확인] 생성 없이 입력 단계에서: FaceID ArcFace가 보호 얼굴을 원본 얼굴과 얼마나 같게 보는가
# FaceID 방식 그대로 (여백 → 얼굴 찾기 → 5점 정렬 → onnx). 낮을수록 FaceID가 다른 사람으로 읽음
def fid_real(crop_bgr):
    M = fid_matrix(crop_bgr)
    if M is None:
        return None
    a = cv2.cvtColor(cv2.warpAffine(crop_bgr, M, (112, 112), borderValue=0.0), cv2.COLOR_BGR2RGB)
    a = ((a.astype(np.float32) / 255 - 0.5) / 0.5).transpose(2, 0, 1)[None]
    return F.normalize(torch.from_numpy(sess.run(None, {sess.get_inputs()[0].name: a})[0]), dim=1)

for n in names:
    clean = fid_real(cv2.imread(f'{RESULTS}/robust_eps0/{n}_eps0.png'))
    row = []
    for tag in ['eps5_n4_seed0', 'ens_eps5_n4_seed0']:   # Week 6 보호(pilot) vs ensemble
        e = fid_real(cv2.imread(f'{RESULTS}/{tag}/{n}_eps5.png'))
        row.append('얼굴 못 찾음' if e is None else f'{F.cosine_similarity(clean, e).item():.3f}')
    print(f'{n} | FaceID ArcFace 입력 유사도  Week 6 보호 {row[0]}  →  ensemble {row[1]}')


#### Ensemble 반복 실행 (노이즈 seed 1, 2)
- seed 0 결과 (`docs/weekly/week7.md`): SimSwap ArcFace -0.002 (4장 성공), FaceID 입력 0.015, FaceID 생성 ArcFace 0.083 / FaceNet 0.510
- Week 6 ④와 같은 방식: 랜덤 출발과 EOT 변형만 바뀜 → 결과가 seed에 따라 얼마나 흔들리는지 확인
- 위의 `[ensemble 공격]` 셀(함수 정의)과 `[사전 확인]` 셀(`fid_real`)을 먼저 실행해야 한다
- 끝나면 Drive `week6/ens_eps5_n4_seed{0,1,2}/` 세 폴더를 Mac `data/week6/`로 내려받는다 (SimSwap 결과 FaceNet 채점용)


In [ ]:
# [ensemble 반복] 노이즈 seed 1, 2 (seed 0은 위에서 실행함)
for seed in [1, 2]:
    run(f'ens_eps5_n4_seed{seed}', 5, attack=pgd_ensemble_eot, seed=seed)

# 입력 단계 FaceID ArcFace 유사도 (보호 얼굴 vs 보호 안 한 얼굴), seed별
for n in names:
    clean = fid_real(cv2.imread(f'{RESULTS}/robust_eps0/{n}_eps0.png'))
    row = []
    for seed in [0, 1, 2]:
        e = fid_real(cv2.imread(f'{RESULTS}/ens_eps5_n4_seed{seed}/{n}_eps5.png'))
        row.append('얼굴 못 찾음' if e is None else f'{F.cosine_similarity(clean, e).item():.3f}')
    print(f'{n} | FaceID ArcFace 입력 유사도 seed 0/1/2: {" ".join(row)}')


## Week 8 ③④ 사진 전체 보호 + 처리 속도
- 지금까지는 얼굴만 잘라낸 224×224 사진에만 노이즈를 넣었다. 서비스는 **사진 전체**를 받아 **보호된 사진 전체**를 돌려줘야 한다
- 방식 A (가장 단순): 사진 전체 → SimSwap 방식으로 얼굴 찾기·정렬(224) → ensemble 노이즈 → **노이즈만** 원래 위치·크기로 되돌려 붙이기
  - 얼굴 crop 자체를 붙이지 않고 노이즈만 붙여서, 사진의 원래 화질은 그대로 둔다
  - 걱정: 내 사진의 얼굴은 430 ~ 1120픽셀이라 224 노이즈가 2 ~ 5배 늘어나 붙는다. 딥페이크가 다시 줄여서 읽을 때 노이즈가 살아남는지 확인해야 한다
- 평가: 보호된 사진 전체를 그대로 딥페이크에 넣는다 (여백 붙이기 없이, 실제 사용과 같음)
- 결과: Drive `week8/full_eps5_seed0/`. FaceID 생성은 `ipadapter_faceid_eval.ipynb`의 Week 8 셀
- 실행 순서: `준비 1` → `준비 2` → `[도구 함수]` → `[2단계] EOT` → `[ensemble 준비 1]` → `[ensemble 준비 3]` → `[ensemble 공격]`(맨 끝 `ens = run(...)` 줄은 지워도 됨) → `[사전 확인]` → 아래 셀


In [ ]:
# [Week 8 ③] 사진 전체 보호 함수 (방식 A: 노이즈만 되돌려 붙이기) + 단계별 시간 측정
import time

def protect_full(photo_bgr, eps=5/255, steps=30, attack=pgd_ensemble_eot):
    # 사진 전체(BGR) → 보호된 사진 전체(BGR). 얼굴을 못 찾으면 None
    t0 = time.time()
    found = app.get(photo_bgr, crop_size)            # SimSwap 방식: 얼굴 찾기 → 224 정렬 crop + 변환 M (원본 → crop)
    if found is None:
        return None, None, None
    crops, mats = found
    face, M = to_tensor(crops[0]), mats[0]
    t1 = time.time()

    adv = attack(face, eps, steps)                   # Week 7 ensemble 노이즈 (224 얼굴에서)
    torch.cuda.synchronize()
    t2 = time.time()

    # 노이즈만 원래 사진 좌표로 되돌림 (M의 역변환). 얼굴 바깥은 0
    delta = (adv - face)[0].permute(1, 2, 0).cpu().numpy()[..., ::-1].copy()   # RGB → BGR, 224×224×3
    H, W = photo_bgr.shape[:2]
    Minv = cv2.invertAffineTransform(M)
    delta_big = cv2.warpAffine(delta, Minv, (W, H), flags=cv2.INTER_LINEAR, borderValue=0)
    out = np.clip(photo_bgr.astype(np.float32) / 255 + delta_big, 0, 1)
    out = (out * 255).round().astype(np.uint8)
    t3 = time.time()
    region = cv2.warpAffine(np.ones((crop_size, crop_size), np.float32), Minv, (W, H)) > 0.5   # 노이즈가 들어간 영역 (화질 측정용)
    return out, {'찾기·정렬': t1 - t0, '노이즈': t2 - t1, '붙이기': t3 - t2, '전체': t3 - t0}, region

def psnr_np(a, b, mask=None):
    d = (a.astype(np.float32) - b.astype(np.float32)) ** 2
    mse = d[mask].mean() if mask is not None else d.mean()
    return 10 * np.log10(255 ** 2 / max(mse, 1e-10))

print('사진 전체 보호 함수 준비 완료')


In [ ]:
# [Week 8 ③④ 실행] 내 원본 사진 4장 → 보호된 사진 전체 → SimSwap 채점 + FaceID 입력 확인 + 시간·화질
W8 = f'{DRIVE}/week8/full_eps5_seed0'
os.makedirs(W8, exist_ok=True)
random.seed(0); torch.manual_seed(0)

def fid_full(img_bgr):
    # FaceID 방식 그대로: 사진 전체에서 얼굴 찾기 → 5점 정렬 112 → onnx (여백 없이)
    M = fid_matrix(img_bgr, pad=0)
    if M is None:
        return None
    a = cv2.cvtColor(cv2.warpAffine(img_bgr, M, (112, 112), borderValue=0.0), cv2.COLOR_BGR2RGB)
    a = ((a.astype(np.float32) / 255 - 0.5) / 0.5).transpose(2, 0, 1)[None]
    return F.normalize(torch.from_numpy(sess.run(None, {sess.get_inputs()[0].name: a})[0]), dim=1)

for name in names:
    photo = cv2.imread(f'{PHOTOS}/{name}.jpg')
    me = arc_embed_img(photo)
    prot, times, region = protect_full(photo)
    if prot is None:
        print(f'{name} | 얼굴 못 찾음'); continue
    cv2.imwrite(f'{W8}/{name}.png', prot)                      # PNG (무손실)

    # 1) SimSwap: 보호된 사진 전체를 그대로 source로 (여백 붙이기 없이)
    lat = arc_embed_img(prot)
    sw = None
    if lat is not None:
        swap_with_latent(lat, f'{W8}/{name}_simswap.jpg')
        sw = (arc_embed_img(cv2.imread(f'{W8}/{name}_simswap.jpg')) * me).sum().item()
    # 2) 입력 단계 신원 (보호 사진 전체 vs 원본 사진 전체)
    arc_in = (lat * me).sum().item() if lat is not None else None
    a, b = fid_full(photo), fid_full(prot)
    fid_in = None if a is None or b is None else F.cosine_similarity(a, b).item()
    # 3) 화질: 사진 전체, 노이즈가 들어간 영역만
    t = ' '.join(f'{k} {v:.1f}s' for k, v in times.items())
    fmt = lambda v: '-' if v is None else f'{v:.3f}'
    print(f'{name} {photo.shape[1]}×{photo.shape[0]} | SimSwap 결과 vs 나 {fmt(sw)} ({judge(sw)}) '
          f'| 입력: SimSwap ArcFace {fmt(arc_in)}, FaceID ArcFace {fmt(fid_in)} '
          f'| PSNR 전체 {psnr_np(photo, prot):.1f}dB, 얼굴 영역 {psnr_np(photo, prot, region):.1f}dB | {t}')
